In [1]:
import os
os.environ["POLYGON_API_KEY"] = "yIgSEdpRrHsyYoA3pyHNBx5_XipyGMBK"   # <-- replace, then run this cell first


# Cup Coffee — Data Cleaning, step by step

Run each cell top-to-bottom and **watch the data change**. The goal is to *see* exactly what
"cleaning" does to the raw market data before it ever reaches the strategy.

Pipeline we'll walk:
`raw Polygon bars → keep regular hours → downsample → the trade table → drop tiny-stop noise →
fix reverse-split ghost prices → flag commodity names → charge real costs → the clean set.`

> Run this from the project root so `import research_data` works. The bar cells read the
> on-disk cache (need `POLYGON_API_KEY` in your env; cached, so no real download).

In [2]:
import os, json
import pandas as pd
from datetime import date as Date, datetime, timezone
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

DATA = "data"
def load_jsonl(p): return [json.loads(l) for l in open(p) if l.strip()]
def key(e):       # the join key used across every data file
    return f'{e["symbol"]}|{e["day"]}|{e["timeframe"]}|{e.get("breakout_idx")}|{e.get("handle_num")}'
print("setup ready")

setup ready


## 1. Raw Polygon bars → clean candles  (`data_layer.py` + `research_data.py`)

Polygon hands us **every** minute of the session — including pre-market and after-hours. Step 1
of cleaning is to keep only **regular trading hours (09:30–16:00 ET)**.

In [3]:
from research_data import ResearchData, ET
rd = ResearchData(os.environ["POLYGON_API_KEY"])
SYM, DAY = "AVXL", Date(2021, 6, 28)              # our running example trade

raw = rd._minute_raw(SYM, DAY)                    # RAW cached Polygon rows (all sessions)
rawdf = pd.DataFrame(raw)
rawdf["time_ET"] = [datetime.fromtimestamp(t/1000, tz=timezone.utc).astimezone(ET).strftime("%H:%M")
                    for t in rawdf["t"]]
print(f"RAW rows from Polygon: {len(rawdf)}   (first {rawdf['time_ET'].iloc[0]}  last {rawdf['time_ET'].iloc[-1]})")
rawdf[["time_ET","o","h","l","c","v"]].head()

RAW rows from Polygon: 605   (first 05:14  last 19:58)


,time_ET,o,h,l,c,v
0,05:14,24.96,24.96,24.96,24.96,111
1,07:00,25.28,26.00,25.28,25.99,1513
2,07:02,25.99,26.00,25.99,26.00,255
3,07:03,26.00,26.00,26.00,26.00,700
4,07:04,25.55,25.55,25.55,25.55,219


In [4]:
# After the regular-hours filter (this is what research_data.bars() returns):
one = rd.bars(SYM, DAY, "1min")
print(f"RAW {len(rawdf)} rows  ->  REGULAR-HOURS {len(one)} one-minute bars")
print(f"   (dropped {len(rawdf)-len(one)} pre/after-hours bars; kept {one.ts[0].time()}–{one.ts[-1].time()})")

RAW 605 rows  ->  REGULAR-HOURS 389 one-minute bars
   (dropped 216 pre/after-hours bars; kept 09:30:00–15:59:00)


### Downsampling: 1-min → 5-min
Each 5-minute candle = **open** of the first minute, **max high**, **min low**, **close** of the
last minute, **summed volume**. Watch five 1-min bars collapse into one.

In [5]:
five = rd.bars(SYM, DAY, "5min")
print(f"{len(one)} one-min bars  ->  {len(five)} five-min bars\n")
first5 = pd.DataFrame({"time":[t.strftime('%H:%M') for t in one.ts[:5]],
                       "o":one.o[:5], "h":one.h[:5], "l":one.l[:5], "c":one.c[:5], "v":one.v[:5]})
print("the first FIVE 1-min bars:"); display(first5)
collapsed = pd.DataFrame({"time":[five.ts[0].strftime('%H:%M')], "o":[five.o[0]], "h":[five.h[0]],
                          "l":[five.l[0]], "c":[five.c[0]], "v":[five.v[0]]})
print("collapse to ONE 5-min bar  (open=first, high=max, low=min, close=last, vol=sum):")
display(collapsed)

389 one-min bars  ->  78 five-min bars

the first FIVE 1-min bars:


,time,o,h,l,c,v
0,09:30,29.7000,29.87,29.4000,29.5198,276721
1,09:31,29.5900,29.68,29.2300,29.3900,128195
2,09:32,29.4882,30.20,29.3829,29.8000,251128
3,09:33,29.7400,30.50,29.7203,30.3700,144541
4,09:34,30.4150,30.46,29.9600,30.2533,99571


collapse to ONE 5-min bar  (open=first, high=max, low=min, close=last, vol=sum):


,time,o,h,l,c,v
0,09:30,29.7,30.5,29.23,30.2533,900156


## 2. The trade pile as a table
`events.jsonl` is one row per detected trade. Load it into a DataFrame — this is the raw,
labeled dataset everything else operates on.

In [6]:
events = load_jsonl(f"{DATA}/events.jsonl")
ev = pd.DataFrame(events)
print("trades in the pile:", len(ev))
ev[["symbol","day","timeframe","reason","size_tier","entry_price","stop_price","risk_R","outcome","pnl_R"]].head(8)

trades in the pile: 10775


,symbol,day,timeframe,reason,size_tier,entry_price,stop_price,risk_R,outcome,pnl_R
0,NASDAQ:QQQ,2021-06-28,1min,whitelist,index,352.9072,352.7900,0.1172,-1,-1.0
1,AMEX:SPY,2021-06-28,1min,whitelist,index,426.4900,426.4316,0.0584,-1,-1.0
2,AMEX:SPY,2021-06-28,1min,whitelist,index,426.4100,426.3400,0.0700,-1,-1.0
3,BEAM,2021-06-28,5min,momentum_gap,large,109.9400,109.1801,0.7599,-1,-1.0
4,AVXL,2021-06-28,1min,momentum_gap,mid,28.2600,28.0301,0.2299,-1,-1.0
5,AVXL,2021-06-28,5min,momentum_gap,mid,29.7900,29.4300,0.3600,-1,-1.0
6,NTLA,2021-06-28,2min,momentum_gap,large,132.8100,131.5000,1.3100,-1,-1.0
7,NASDAQ:QQQ,2021-06-29,1min,whitelist,index,353.7000,353.5500,0.1500,-1,-1.0


## 3. Cleaning #1 — drop the tiny-stop noise
`R = entry − stop`. As a % of price, a *tiny* stop (handle ≈ 0) makes nonsense R-multiples and
gets crushed by fixed costs. Look at the distribution, then the offenders.

In [7]:
ev["stop_pct"] = ev["risk_R"] / ev["entry_price"] * 100
print("stop, as % of price — distribution:")
print(ev["stop_pct"].describe()[["min","25%","50%","75%","max"]])
print("\nthe tiniest stops (sub-noise — mostly index trades):")
ev.sort_values("stop_pct")[["symbol","day","entry_price","risk_R","stop_pct"]].head(6)

stop, as % of price — distribution:
min     0.000072
25%     0.051407
50%     0.149958
75%     0.455354
max    11.757576
Name: stop_pct, dtype: float64

the tiniest stops (sub-noise — mostly index trades):


,symbol,day,entry_price,risk_R,stop_pct
3337,CHSN,2023-05-10,13840.01,0.01,0.000072
1016,AUUD,2021-12-08,7003.16,0.01,0.000143
1014,AUUD,2021-12-08,6602.30,0.01,0.000151
5140,HOLO,2024-04-04,3152.01,0.01,0.000317
6311,LGMK,2024-10-07,2456.26,0.01,0.000407
5567,NVVE,2024-06-05,572.01,0.01,0.001748


In [8]:
MIN_STOP = 0.25                                   # %
keep = ev["stop_pct"] >= MIN_STOP
print(f"min-stop {MIN_STOP}% filter:  {len(ev)}  ->  {keep.sum()} trades   (dropped {len(ev)-keep.sum()})")

min-stop 0.25% filter:  10775  ->  4143 trades   (dropped 6632)


## 4. Cleaning #2 — fix the reverse-split *ghost* prices  (the big one)
Polygon prices are split-**adjusted**, so a stock that later reverse-split shows a wildly inflated
history. `realprice.json` (from `enrich_real_price.py`) recovers the **real** price. Watch the
before/after.

In [9]:
RP = json.load(open(f"{DATA}/realprice.json"))
ev["real_price"] = ev.apply(lambda r: RP.get(key(r), {}).get("real_price", r["entry_price"]), axis=1)
ev["real_risk"]  = ev.apply(lambda r: RP.get(key(r), {}).get("real_risk",  r["risk_R"]),       axis=1)
ghosts = ev[ev["entry_price"] > 2000].sort_values("entry_price", ascending=False)
print(f"{len(ghosts)} trades carry split-inflated prices. The worst offenders (adjusted vs REAL):")
ghosts[["symbol","day","entry_price","real_price","risk_R","real_risk"]].head(8)

460 trades carry split-inflated prices. The worst offenders (adjusted vs REAL):


,symbol,day,entry_price,real_price,risk_R,real_risk
3865,ADTX,2023-08-18,4.149360e+09,17.000,1.064189e+07,0.043600
4097,SMX,2023-10-10,3.306414e+09,2.770,1.473113e+07,0.012341
4741,SMX,2024-02-07,5.610377e+08,0.470,1.326866e+07,0.011116
5744,ADTX,2024-07-17,5.442984e+08,2.230,4.881600e+06,0.020000
682,EJH,2021-10-26,3.775000e+08,3.020,3.750000e+06,0.030000
681,EJH,2021-10-26,3.768750e+08,3.015,4.375000e+06,0.035000
680,EJH,2021-10-26,3.637500e+08,2.910,6.250000e+05,0.005000
5888,ADTX,2024-08-07,3.417120e+08,1.400,4.271400e+06,0.017500


In [10]:
# Why it matters: a fixed fee divided by the FAKE dollar-risk looks ~free; on the REAL risk it's brutal.
FEE = (2.0 + 2*0.3) / 100.0                       # $0.026: 2c slippage + 0.6c round-trip commission
ev["cost_adj_R"]  = FEE / ev["risk_R"]            # WRONG — cost on the inflated price
ev["cost_real_R"] = FEE / ev["real_risk"]         # RIGHT — cost on the real price
print("cost in R: on the fake price (≈0) vs the real price (huge):")
ghosts2 = ev[ev["entry_price"] > 2000]
ghosts2[["symbol","entry_price","real_price","cost_adj_R","cost_real_R"]].head(6)

cost in R: on the fake price (≈0) vs the real price (huge):


,symbol,entry_price,real_price,cost_adj_R,cost_real_R
13,DBGI,775000.01,6.20,0.000003,0.433333
14,DBGI,678750.01,5.43,0.000006,0.742857
15,XELA,8680.01,2.17,0.000217,0.866580
16,XELA,8320.01,2.08,0.000650,2.599220
17,XELA,8040.01,2.01,0.000353,1.412813
18,XELA,8360.01,2.09,0.000325,1.299805


In [11]:
MIN_PRICE = 15                                    # $, on the REAL price
keep_price = ev["real_price"] >= MIN_PRICE
print(f"min REAL price ${MIN_PRICE} filter:  {len(ev)}  ->  {keep_price.sum()}   (dropped {len(ev)-keep_price.sum()} penny names)")

min REAL price $15 filter:  10775  ->  8959   (dropped 1816 penny names)


## 5. Cleaning #3 — flag commodity-sector names  (`enrich_commodity.py`, SIC-based)

In [12]:
COMM = json.load(open(f"{DATA}/commodity.json"))
ev["commodity"] = ev["symbol"].str.split(":").str[-1].map(lambda s: COMM.get(s, False))
print("commodity-sector trades flagged:", int(ev["commodity"].sum()), "of", len(ev))
ev[ev["commodity"]][["symbol","day","reason"]].drop_duplicates("symbol").head(10)

commodity-sector trades flagged: 85 of 10775


,symbol,day,reason
713,METC,2021-10-29,earnings_gap
1287,CNR,2022-02-14,momentum_gap
1382,MXC,2022-03-08,earnings_gap
1383,PED,2022-03-08,earnings_gap
1497,HYMC,2022-03-28,momentum_gap
1559,LBRT,2022-04-21,earnings_gap
2422,AA,2022-11-04,momentum_gap
2780,HPK,2023-01-24,earnings_gap
3278,IEP,2023-05-05,momentum_gap
3846,EDBL,2023-08-10,momentum_gap


## 6. Charge real costs, then assemble the CLEAN set
Apply every screen (min-stop + min-real-price), then compute **net** R at a take-profit using the
full-path labels (`mined_table.json`) minus the **real** cost.

In [13]:
TAB = {r["key"]: r for r in json.load(open(f"{DATA}/mined_table.json")) if "key" in r}
clean = ev[(ev["stop_pct"] >= MIN_STOP) & (ev["real_price"] >= MIN_PRICE)].copy()
def net_at(r, k):
    m = TAB.get(key(r))
    return None if not m else m["realized_R"][str(k)] - FEE / r["real_risk"]
clean["net_8R"] = clean.apply(lambda r: net_at(r, 8), axis=1)

print(f"RAW pile        : {len(ev):>6} trades")
print(f"CLEAN pile      : {len(clean):>6} trades  (after min-stop {MIN_STOP}% + min-real-price ${MIN_PRICE})")
print(f"\nCLEAN @ 8R, net of REAL costs: total {clean['net_8R'].sum():+.0f}R | "
      f"avg {clean['net_8R'].mean():+.3f}R/trade | win {100*(clean['net_8R']>0).mean():.0f}%")

RAW pile        :  10775 trades
CLEAN pile      :   2457 trades  (after min-stop 0.25% + min-real-price $15)

CLEAN @ 8R, net of REAL costs: total +539R | avg +0.219R/trade | win 22%


## Recap — the cleaning pipeline
1. **Raw bars → regular hours** (drop pre/after-market)
2. **Downsample** 1→5-min (open/maxH/minL/close/sumV)
3. **Trade table** (one row per detected trade)
4. **Tiny-stop screen** — remove sub-noise R-multiples
5. **Real-price fix** — undo split inflation; recover true price & dollar-risk
6. **Commodity flag** — SIC-based sector tag
7. **Real-cost net R** — the honest, analyzable dataset

Every later step (factor mining, the dashboard, ML) runs on the **clean** set, not the raw pile.
Change `MIN_STOP`, `MIN_PRICE`, or `FEE` above and re-run to watch the clean set move.